# 

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import duckdb
import shap
import json
import os
from tqdm.auto import tqdm
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging

# -----------------------------
# 환경 설정
# -----------------------------
warnings.filterwarnings('ignore')
os.environ['LIGHTGBM_LOG_LEVEL'] = '-1'
logging.getLogger("tqdm").setLevel(logging.ERROR)


# -----------------------------
# 유틸 함수
# -----------------------------
def get_exclusion_list(paths):
    exclude_set = set()
    for path in paths:
        if not os.path.exists(path):
            print(f"⚠️ 제외 설정 파일 없음: {path}")
            continue
        with open(path, 'r', encoding='utf-8') as f:
            cfg = json.load(f)
            for key in cfg:
                if isinstance(cfg[key], list):
                    exclude_set.update(cfg[key])
    return list(exclude_set)


def check_gpu():
    try:
        lgb.train({'device': 'gpu', 'verbose': -1},
                  lgb.Dataset(np.zeros((1, 1)), label=[0]))
        return True
    except:
        return False


def train_lgb_cv(X, y, model, cv):
    scores = []
    importances = []

    for tr_idx, val_idx in tqdm(list(cv.split(X, y)), desc="🚀 CV"):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric='average_precision',
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        pred = model.predict_proba(X_val)[:, 1]
        score = average_precision_score(y_val, pred)

        scores.append(score)
        importances.append(model.feature_importances_)

    return scores, importances


def build_importance_df(X, cv_importances, final_model):
    cv_imp = np.mean(cv_importances, axis=0)
    final_imp = final_model.feature_importances_

    df = pd.DataFrame({
        'feature': X.columns,
        'cv_importance': cv_imp,
        'final_importance': final_imp
    })

    df['importance'] = 0.7 * df['cv_importance'] + 0.3 * df['final_importance']
    return df.sort_values(by='importance', ascending=False)


def plot_importance(feat_imp):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    sns.barplot(
        x='importance', y='feature',
        data=feat_imp.head(10),
        ax=axes[0]
    )
    axes[0].set_title("Top 10 Features")

    sns.barplot(
        x='importance', y='feature',
        data=feat_imp.tail(10),
        ax=axes[1]
    )
    axes[1].set_title("Bottom 10 Features")

    plt.tight_layout()
    plt.show()


def run_shap(model, X_test):
    print("\n🔮 SHAP 분석 중...")

    X_sample = X_test.sample(200, random_state=42)

    X_proc = X_sample.copy()
    num_cols = X_proc.select_dtypes(include=[np.number]).columns
    X_proc[num_cols] = X_proc[num_cols].astype('float32')

    explainer = shap.TreeExplainer(model)
    sv = explainer(X_proc)

    shap.plots.beeswarm(sv, max_display=20)
    shap.plots.bar(sv, max_display=20)

    shap_values = sv.values
    shap_importance = np.abs(shap_values).mean(axis=0)

    shap_imp_df = pd.DataFrame({
        'feature': X_proc.columns,
        'importance': shap_importance
    }).sort_values(by='importance', ascending=False)

    return shap_imp_df


# -----------------------------
# 메인 파이프라인
# -----------------------------
def main():

    train_path = "../data/fs_data/fs_train.parquet"
    test_path = "../data/fs_data/fs_validation.parquet"

    REMOVE_LISTS = [
        r'..\config\remove_features_v1.json',
        r'..\config\remove_features_v2.json',
        # r'..\config\raw_features.json'
    ]

    print("📦 데이터 로드 중...")
    train_df = duckdb.query(f"SELECT * FROM '{train_path}'").df()
    test_df = duckdb.query(f"SELECT * FROM '{test_path}'").df()

    print(f"✅ Train: {train_df.shape}, Test: {test_df.shape}")

    # GPU 확인
    has_gpu = check_gpu()
    print(f"🖥️ Device: {'GPU' if has_gpu else 'CPU'}")

    # 컬럼 제거
    base_drop = ['failure', 'date', 'serial_number']
    raw_drop = [
        # 'smart_197_raw', 
        # 'smart_198_raw', 
        # 'smart_187_raw',
        # # 's198_damaged',
        ]

    json_drop = get_exclusion_list(REMOVE_LISTS)
    drop_cols = list(set(base_drop + raw_drop + json_drop))

    print(f"🚫 JSON 제외 피처: {len(json_drop)}개")

    X_train = train_df.drop(columns=drop_cols, errors='ignore')
    y_train = train_df['failure']

    X_test = test_df.drop(columns=drop_cols, errors='ignore')
    y_test = test_df['failure']

    print(f"📊 Feature 수: {X_train.shape[1]}")

    # 모델
    model = LGBMClassifier(
        objective='binary',
        importance_type='gain',
        class_weight='balanced',
        max_depth=6,
        num_leaves=40,
        n_estimators=200,
        learning_rate=0.07,
        device='gpu' if has_gpu else 'cpu',
        random_state=42,
        bagging_seed=42,
        feature_fraction_seed=42,
        data_random_seed=42,
        n_jobs=-1,
        verbose=-1
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # CV 학습
    cv_scores, cv_importances = train_lgb_cv(X_train, y_train, model, cv)

    # 전체 학습
    model.fit(X_train, y_train)

    test_pred = model.predict_proba(X_test)[:, 1]
    test_score = average_precision_score(y_test, test_pred)

    print(f"\n📊 CV PR-AUC: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
    print(f"🎯 Test PR-AUC: {test_score:.4f}")

    # 중요도
    feat_imp = build_importance_df(X_train, cv_importances, model)

    print("\n🏆 Top Features:")
    print(feat_imp.head(20).to_string(index=False))

    plot_importance(feat_imp)

    # SHAP
    run_shap(model, X_test)


# -----------------------------
# 실행
# -----------------------------
if __name__ == "__main__":
    main()

📦 데이터 로드 중...
✅ Train: (299442, 310), Test: (682962, 310)
🖥️ Device: GPU
🚫 JSON 제외 피처: 141개
📊 Feature 수: 167


🚀 CV:   0%|          | 0/5 [00:00<?, ?it/s]


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: serial_number: str